In [0]:
from pyspark.sql.functions import col, trim, lower, upper, current_timestamp, translate, regexp_replace, to_date, to_timestamp
from delta.tables import DeltaTable

# 1. Parámetros (solo el ambiente es necesario ahora)
dbutils.widgets.text("env", "dev", "Ambiente")
ambiente = dbutils.widgets.get("env")

# 2. Diccionario con todas las tablas y sus llaves primarias
tablas_silver = {
    "categorias": "id_categoria",
    "clientes": "id_cliente",
    "empleados": "id_empleado",
    "productos": "id_producto",
    "proveedores": "id_proveedor",
    "subcategorias": "id_subcategoria",
    "sucursales": "id_sucursal",
    "ordenes_venta": "id_orden",
    "ordenes_venta_detalle": "id_detalle",
    "facturas": "id_factura",
    "movimientos_inventario": "id_movimiento"
}

# 3. Bucle de limpieza universal
for nombre_tabla, llave_primaria in tablas_silver.items():
    tabla_origen = f"ferreteria_{ambiente}.bronze.{nombre_tabla}"
    tabla_destino = f"ferreteria_{ambiente}.silver.{nombre_tabla}"

    print(f"\n==== Procesando limpieza Plata: {nombre_tabla} | PK: {llave_primaria} ====")

    try:
        df = spark.read.table(tabla_origen)
    except Exception as e:
        print(f"[!] La tabla {tabla_origen} no existe en Bronce aún. Saltando...")
        continue

    # A. Eliminar columnas técnicas de Bronce
    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    # B. Convertir todos los nombres de columnas a minúsculas
    df = df.select([col(c).alias(c.lower()) for c in df.columns])

    # C. Identificar columnas de texto (String)
    columnas_texto = [f.name for f in df.schema.fields if f.dataType.typeName() == 'string']

    for c in columnas_texto:
        if c == 'rfc':
            df = df.withColumn(c, upper(trim(col(c)))) 
        else:
            df = df.withColumn(c, lower(trim(col(c)))) 
        
        df = df.withColumn(c, translate(col(c), 'áéíóúü', 'aeiouu'))
        df = df.withColumn(c, regexp_replace(col(c), r'[^a-zA-Z0-9\s\.\-@_]', ''))

    # D. Tratamiento de Fechas y Nulos
    for c in df.columns:
        if "fecha" in c:
            if "hora" in c or "timestamp" in c:
                df = df.withColumn(c, to_timestamp(col(c)))
            else:
                df = df.withColumn(c, to_date(col(c)))
        
        if c in columnas_texto:
            df = df.fillna('n/d', subset=[c])

    # E. Eliminar Duplicados completos
    df = df.dropDuplicates()

    # F. Columna de Auditoría
    df = df.withColumn("fecha_carga_plata", current_timestamp())

    # 4. GUARDADO EN UNITY CATALOG (Upsert / MERGE)
    tabla_existe = spark.catalog.tableExists(tabla_destino)

    if not tabla_existe:
        print(f"[*] La tabla {tabla_destino} no existe. Ejecutando carga inicial (Overwrite)...")
        df.write.format("delta").mode("overwrite").saveAsTable(tabla_destino)
        print(f"[OK] {nombre_tabla} creada con éxito en la capa Plata.")
    else:
        print(f"[*] La tabla {tabla_destino} ya existe. Ejecutando MERGE (Upsert)...")
        tabla_delta = DeltaTable.forName(spark, tabla_destino)
        
        (tabla_delta.alias("destino")
         .merge(
             df.alias("origen"),
             f"destino.{llave_primaria} = origen.{llave_primaria}"
         )
         .whenMatchedUpdateAll()     
         .whenNotMatchedInsertAll()  
         .execute()
        )
        print(f"[OK] {nombre_tabla} fusionada con éxito.")